<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/03_ajuste_fino_con_tus_textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 03 · Ajuste fino (*fine-tuning*) de un modelo preentrenado con tus textos

**Objetivo:** tomar un modelo pequeño que ya «sabe» lenguaje y adaptarlo al estilo de tus textos.

Entrenar desde cero (cuaderno 02) necesita muchísimo texto. Es mucho más eficaz partir de un modelo
**preentrenado** (que ya ha leído miles de millones de palabras) y **ajustarlo** con nuestro corpus.

- **Modelo base:** `Qwen/Qwen2.5-0.5B-Instruct` (494 millones de parámetros, buen español).
- **LoRA** (*Low-Rank Adaptation*): en lugar de cambiar todos los parámetros, añadimos unas pocas
  matrices pequeñas y solo entrenamos esas (≈ 2 % del total). Ahorra memoria y tiempo y el resultado
  ocupa solo unos MB.

⏱️ **≈ 25 min** (martes 15:50–16:15). Cada participante entrena en **su propia GPU T4 de Colab**:
300 pasos ≈ 4–6 min (como máximo 8 min: después se para solo).

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro). Puedes repetirla sin problema.
import os, sys, importlib.util
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "/content/llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio (solo la primera vez) y entramos en su carpeta
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    # Colab ya trae torch, transformers, peft, etc.: solo instalamos lo que falte,
    # sin tocar las versiones preinstaladas
    paquetes = {"transformers": "transformers", "tokenizers": "tokenizers", "peft": "peft",
                "accelerate": "accelerate", "yaml": "pyyaml", "networkx": "networkx", "tqdm": "tqdm"}
    faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
    if faltan:
        !pip install -q {" ".join(faltan)}
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
# El código del taller (carpeta src/dhllm) se importa directamente, sin reiniciar
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## 0. ¿Tenemos GPU?

In [ ]:
from dhllm import taller
taller.comprobar_gpu();

## ⚙️ Ajustes
Los textos se toman de `config.yaml` (cuaderno 01). El resto de ajustes, en la sección `finetune`.

In [ ]:
AJUSTES = {"pasos": 300, "ritmo_aprendizaje": 2e-4}
USAR_COPIA = False   # True = sin GPU o sin tiempo: usar el adaptador ya entrenado (backups/, solo textos de ejemplo)
INICIOS = ["En un lugar de la Mancha", "Sancho Panza le dijo a su amo", "Yo, señor, nací en"]

## 1. Cargar el modelo base y tus textos
Los textos se cortan en trozos de 256 tokens.

In [ ]:
modelo, datos = taller.preparar_ajuste_fino(AJUSTES)

## 2. ¿Qué escribe el modelo ANTES del ajuste?
La **perplejidad** indica, más o menos, entre cuántas palabras «duda» el modelo.

In [ ]:
antes = taller.escribir(modelo, INICIOS)
taller.evaluar(modelo, datos);

## 3. Entrenar con LoRA
Verás una barra de progreso y la pérdida de entrenamiento y validación (deben bajar). Si no hay GPU,
se usa automáticamente la copia de seguridad.

In [ ]:
modelo = taller.ajustar_con_mis_textos(modelo, datos, usar_copia=USAR_COPIA)
taller.evaluar(modelo, datos);

## 4. Comparar ANTES y DESPUÉS

In [ ]:
despues = taller.escribir(modelo, INICIOS)
taller.comparar(antes, despues)

> 🔍 **¿Quieres mirar el código?** [`src/dhllm/taller.py`](../../src/dhllm/taller.py) (función
> `ajustar_con_mis_textos`) y [`src/dhllm/finetune.py`](../../src/dhllm/finetune.py) (LoRA y bucle de entrenamiento).

## 🖐️ Tu turno
1. Añade tus propios `INICIOS`.
2. Entrena con tus propios textos (cuaderno 01: `data/mis_textos`). ¿Reconoces «tu» estilo?
3. ⭐ (Opcional) `AJUSTES = {"modelo_base": "HuggingFaceTB/SmolLM2-135M"}`: más rápido, pero ¿qué tal su español?
4. ⭐ (Opcional, en casa) `"metodo": "full"` entrena todos los parámetros: compara tiempo, memoria y resultado.
5. **Para pensar:** si ajustas un modelo con las cartas de una persona, ¿de quién es el texto que genera?

> 🚀 **Para ir más lejos:** el extra `notebooks/extras/E1_instrucciones_sft.ipynb` enseña a un modelo a
> responder preguntas (SFT). [Unsloth](https://github.com/unslothai/unsloth) ofrece cuadernos de Colab
> muy optimizados para ajustar modelos más grandes (1–8 mil millones de parámetros) con QLoRA en una T4.